# 什么因素能够预测创意自我效能感？

这是一个教学式、可运行的分析 Notebook。研究对象是创意自我效能感（Creative Self-Efficacy, CSE），机器学习只是用来检验样本外预测和比较不同领域信息的工具。

研究问题：

1. 心理、行为、背景和环境特征整体能在多大程度上预测 CSE？
2. 哪些构念具有最稳定的条件预测信息？
3. 不同领域包含多少重叠或可替代的预测信息？

> 本研究是横断面、自陈问卷研究。预测重要性不等于因果作用。

## 1. 环境与路径

Notebook 可以从仓库根目录或 `notebooks/` 目录启动。下面的代码会自动寻找项目根目录。

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('项目根目录：', ROOT)
warnings.filterwarnings('ignore', message='Found unknown categories')
sns.set_theme(style='whitegrid')

## 2. 读取数据并核验结果变量

正式分析脚本中的 `prepare_data()` 会读取 SPSS 文件，并处理独生子女跳题与三个明显异常的兄弟姐妹数。原始数据文件不会被改写。

In [ ]:
from construct_level_cv import (
    prepare_data, THEORY_GROUPS, BACKGROUND, DIGITAL, ATTENTION, FLOW,
    SELF_EVALUATION, FAMILY_ENVIRONMENT, NOMINAL, SEED
)

df, metadata = prepare_data()
y = df['CPSA'].astype(float)
all_features = sum(THEORY_GROUPS.values(), [])

summary = pd.Series({
    '样本量': len(df),
    '原始变量数': df.shape[1] - 2,  # prepare_data 新增两个清理变量
    '构念级预测列数': len(all_features),
    'CPSA均值': y.mean(),
    'CPSA标准差': y.std(ddof=1),
    'CPSA最小值': y.min(),
    'CPSA最大值': y.max(),
})
display(summary.to_frame('数值'))

In [ ]:
# 防止结果泄漏：CPSA 必须等于三个结果题项之和，但这些题项不能作为预测变量。
checks = pd.Series({
    'CPSA = CPSA1 + CPSA2 + CPSA3': bool((df.CPSA == df[['CPSA1','CPSA2','CPSA3']].sum(axis=1)).all()),
    'SMC = SMC1 + ... + SMC9': bool((df.SMC == df[[f'SMC{i}' for i in range(1, 10)]].sum(axis=1)).all()),
    'ADHD = AD + HD': bool((df.ADHD == df.AD + df.HD).all()),
    '结果题项未进入预测集': not any(c in all_features for c in ['CPSA1','CPSA2','CPSA3']),
})
display(checks.to_frame('核验结果'))

## 3. 常见机器学习算法简介

### 均值基准模型
始终预测训练集的平均 CSE。它没有使用任何预测因素，用于判断复杂模型是否真正提供了样本外信息。

### 普通线性回归
将各特征乘以系数后相加。优点是直观；缺点是问卷总分通常高度相关，系数容易不稳定。

### Ridge 回归
在损失函数中增加 L2 惩罚，压缩过大的系数，但通常不会把系数变成零。它特别适合存在共线性的预测数据。

### Lasso 回归
使用 L1 惩罚，可以把部分系数压缩到零，相当于自动选择变量。但当多个变量高度相关时，它选择哪一个可能不稳定。

### Elastic Net
结合 Lasso 和 Ridge，同时兼顾变量选择与相关变量的稳定处理。本研究用内部交叉验证选择惩罚强度，是构念级主模型。

### Random Forest
训练许多不同的决策树并平均预测。它能识别非线性和变量交互，但模型解释比线性模型复杂。

### Extra Trees
与随机森林相似，但切分更加随机，通常训练较快，并可能降低方差。

### Gradient Boosting
按顺序训练模型，每一轮重点修正此前的预测误差。它可以表示复杂关系，但更需要控制过拟合。

### 样条加性模型（Spline GAM）
为每个连续预测变量拟合一条平滑曲线，再把各条曲线相加。它能检查单变量非线性，同时保持较强的可解释性，但默认不表示变量间交互。

### RBF 支持向量回归（RBF-SVR）
通过核函数表示平滑的复杂关系。它对缩放和超参数较敏感，因此本研究只把预先设定参数的SVR作为非线性敏感性比较，不根据测试折反复调参。

复杂算法不必然更好。如果正则化线性模型与树模型表现接近，应优先选择更简单、可解释的模型。

## 4. 在交叉验证内部完成预处理

名义分类变量进行独热编码；数值变量在训练折中填补并标准化。树模型只进行缺失值处理。把这些步骤放进 `Pipeline` 可以防止测试数据提前进入预处理。

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical = [c for c in all_features if c in NOMINAL]
numeric = [c for c in all_features if c not in NOMINAL]

linear_preprocess = ColumnTransformer([
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler()),
    ]), numeric),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
    ]), categorical),
])
tree_preprocess = SimpleImputer(strategy='median', add_indicator=True)

In [ ]:
def linear_pipeline(regressor):
    return Pipeline([
        ('preprocess', linear_preprocess),
        ('model', TransformedTargetRegressor(regressor=regressor, transformer=StandardScaler())),
    ])

models = {
    'Mean baseline': Pipeline([('impute', tree_preprocess), ('model', DummyRegressor())]),
    'Ridge': linear_pipeline(RidgeCV(alphas=np.logspace(-3, 3, 25))),
    'Elastic Net': linear_pipeline(ElasticNetCV(
        l1_ratio=[.05, .2, .5, .8, .95, 1.0], cv=5, max_iter=30000, random_state=SEED)),
    'Random Forest': Pipeline([('impute', tree_preprocess), ('model', RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, max_features=.7, n_jobs=-1, random_state=SEED))]),
    'Extra Trees': Pipeline([('impute', tree_preprocess), ('model', ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=3, max_features=.8, n_jobs=-1, random_state=SEED))]),
    'Gradient Boosting': Pipeline([('impute', tree_preprocess), ('model', HistGradientBoostingRegressor(
        max_iter=300, learning_rate=.05, l2_regularization=1.0, random_state=SEED))]),
}
list(models)

## 5. 五折算法比较

为了让 Notebook 易于运行，这里展示一次五折比较。论文正式结果使用脚本中的5次重复五折外层验证。评价指标：R²越高越好，MAE和RMSE越低越好。

In [ ]:
from sklearn.model_selection import KFold, cross_validate

X = df[all_features]
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}
rows = []
for name, model in models.items():
    result = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({
        '模型': name,
        'R²均值': result['test_r2'].mean(),
        'R²标准差': result['test_r2'].std(ddof=1),
        'MAE': -result['test_mae'].mean(),
        'RMSE': -result['test_rmse'].mean(),
    })
model_comparison = pd.DataFrame(rows).sort_values('R²均值', ascending=False)
display(model_comparison.style.format({'R²均值':'{:.3f}', 'R²标准差':'{:.3f}', 'MAE':'{:.3f}', 'RMSE':'{:.3f}'}))

In [ ]:
ax = sns.barplot(data=model_comparison, y='模型', x='R²均值', color='#4C78A8')
ax.axvline(0, color='black', linewidth=.8)
ax.set(title='构念级模型的五折样本外表现', xlabel='平均测试集 R²', ylabel='')
plt.show()

## 6. 正式重复交叉验证结果

下面直接读取已经由正式脚本生成的25个外层测试折结果。这样既能保持 Notebook 简洁，也能展示论文所用的完整分析。

In [ ]:
phase2 = ROOT / 'results' / 'phase2_construct_cv'
hierarchy = pd.read_csv(phase2 / 'hierarchical_model_summary.csv')
increments = pd.read_csv(phase2 / 'hierarchical_paired_delta_r2.csv')
ablation = pd.read_csv(phase2 / 'leave_one_group_out_delta_r2.csv')

print('分层模型：')
display(hierarchy.round(4))
print('配对增量：')
display(increments.round(4))
print('删除整个领域并重新训练：')
display(ablation.round(4))

### 如何理解删除重训？

删除一个领域后重新训练，允许模型利用其余变量进行补偿，因此更接近该领域有多少信息难以被其他领域替代。各领域损失不能相加，也不能解释成各领域独立解释的方差比例。

## 7. 构念级预测图谱

置换重要性在测试折中随机打乱一个构念，观察固定模型的R²下降。如果一个构念包含多列，所有列使用相同的行排列，保持组内联合分布。

In [ ]:
profile_path = ROOT / 'results' / 'phase4_predictor_profile' / 'CSE_construct_predictor_profile.csv'
profile = pd.read_csv(profile_path)
display(profile[['rank','construct','importance_mean','importance_sd_across_splits',
                 'positive_splits','conditional_direction']].head(15).round(4))

In [ ]:
top = profile.head(12).sort_values('importance_mean')
plt.figure(figsize=(9, 6))
plt.barh(top['construct'], top['importance_mean'], color='#59A14F')
plt.xlabel('测试集R²下降（共同置换重要性）')
plt.ylabel('')
plt.title('创意自我效能感的构念级预测图谱')
plt.tight_layout()
plt.show()

置换重要性回答固定模型有多依赖一个构念；删除重训回答其他变量能否替代它。SMC的置换降幅约为0.432，而删除重训损失约为0.157。不能把0.432表述为SMC独立解释了43.2%的CSE方差。

## 8. 稳健性与测量检查

In [ ]:
phase3 = ROOT / 'results' / 'phase3_robustness'
bootstrap = pd.read_csv(phase3 / 'participant_bootstrap_increment.csv')
cfa = pd.read_csv(phase3 / 'CPSA_SMC_CFA_fit.csv')
smc_sensitivity = pd.read_csv(phase3 / 'SMC_semantic_overlap_sensitivity.csv')

print('固定OOF预测的参与者Bootstrap：')
display(bootstrap.round(4))
print('CPSA与SMC测量模型：')
display(cfa.round(4))
print('排除语义接近的心流题项：')
display(smc_sensitivity.round(4))

## 9. 构念级统一算法基准

为了公平判断非线性模型是否真的更好，正式基准让8种算法使用完全相同的构念特征和25个外层测试折。Ridge和Elastic Net只在各自的训练折内选择正则化强度；其他非线性模型采用预先设定的保守参数。

随机森林的平均R²略高于Elastic Net，但优势只出现在16/25折，折间描述区间跨0。它的MAE则在25/25折更低。因此，树模型可以作为有用的预测敏感性分析，却没有形成足够稳定的R²优势来取代更易解释的Elastic Net主模型。

In [ ]:
phase5 = ROOT / 'results' / 'phase5_algorithm_benchmark'
algorithm_summary = pd.read_csv(phase5 / 'algorithm_summary.csv')
paired_algorithms = pd.read_csv(phase5 / 'paired_performance_vs_elastic_net.csv')
calibration = pd.read_csv(phase5 / 'algorithm_calibration.csv')

display(algorithm_summary.round(4))
display(paired_algorithms[[
    'model', 'delta_r2_vs_elastic_net', 'r2_better_splits',
    'mae_improvement_vs_elastic_net', 'mae_better_splits'
]].round(4))
display(calibration.round(4))

plt.figure(figsize=(9, 5))
plot_data = algorithm_summary.sort_values('r2_mean')
plt.barh(plot_data['model'], plot_data['r2_mean'], color='#4C78A8')
plt.axvline(0, color='black', linewidth=.8)
plt.xlabel('25个外层折的平均样本外R²')
plt.ylabel('')
plt.title('构念级统一算法比较')
plt.tight_layout()

## 10. 深化分析：测量、领域分解与跨模型稳定性

前面的删除重训只衡量一个领域在最后进入完整模型时有多少信息不可被替代。精确Shapley分解进一步评估每个领域在全部可能进入顺序中的平均边际预测效用。与此同时，有序DWLS重新检验CSE与心流的测量区分，跨模型共同置换检验构念排名是否依赖Elastic Net。

In [ ]:
phase6 = ROOT / 'results' / 'phase6_ordinal_measurement'
ordinal_fit = pd.read_csv(phase6 / 'ordinal_DWLS_fit.csv')
ordinal_validity = pd.read_csv(phase6 / 'ordinal_discriminant_validity.csv')

phase7 = ROOT / 'results' / 'phase7_domain_shapley'
domain_shapley = pd.read_csv(phase7 / 'domain_shapley_summary.csv')

phase8 = ROOT / 'results' / 'phase8_cross_model_stability'
rank_agreement = pd.read_csv(phase8 / 'cross_model_rank_agreement.csv')
rank_comparison = pd.read_csv(phase8 / 'construct_rank_comparison.csv')

print('有序题项DWLS测量模型：')
display(ordinal_fit[['model', 'CFI', 'TLI', 'RMSEA', 'SRMR', 'latent_correlation']].round(4))
display(ordinal_validity.round(4))
print('六领域精确Shapley分解：')
display(domain_shapley[['domain', 'shapley_r2_mean', 'share_of_mean_full_model_gain', 'positive_splits']].round(4))
print('跨模型排名一致性：')
display(rank_agreement.round(4))
display(rank_comparison.head(15).round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
shapley_plot = domain_shapley.sort_values('shapley_r2_mean')
axes[0].barh(shapley_plot['domain'], shapley_plot['shapley_r2_mean'], color='#59A14F')
axes[0].set_xlabel('平均Shapley R²贡献')
axes[0].set_title('六领域预测效用分配')
axes[1].scatter(rank_comparison['elastic_net_importance'], rank_comparison['random_forest_importance'], alpha=.75)
for _, row in rank_comparison.head(4).iterrows():
    axes[1].annotate(row['construct'], (row['elastic_net_importance'], row['random_forest_importance']), fontsize=8)
axes[1].set_xlabel('Elastic Net置换重要性')
axes[1].set_ylabel('随机森林置换重要性')
axes[1].set_title('构念重要性的跨模型一致性')
plt.tight_layout()

### 如何综合解释？

心流获得约59%的平均预测效用分配，是最突出的信息来源；自我评价约占19%。数字行为的Shapley贡献约为0.034，但完整模型中的删除损失只有约0.006，说明其多数信息与心理领域共享。Elastic Net与随机森林的前四名完全一致，支持心流、积极自我评价、数字媒体多任务和多动是较稳健的核心预测构念。

有序DWLS双因子模型支持CSE与心流并非同一构念，但完整心流九题模型的RMSEA仍为0.085。删除三个语义接近题项后拟合改善只能作为敏感性证据，不能据此宣称新量表已经得到验证。DWLS Shapley值和置换重要性都不是因果效应。

## 11. 结论

CSE的样本外预测信息分布明显不均衡。Shapley分解显示，特质心流获得约59%的平均预测效用分配，自我评价约占19%；数字行为、注意调节、背景和家庭环境的平均份额较小。删除重训进一步显示，数字行为和环境信息多数可被其他领域替代。

有序DWLS和跨模型稳定性支持主要结论，但完整心流量表仍存在结构异质性，较弱构念的排名也具有模型依赖性。因此，心流的突出贡献必须结合构念重叠和共同方法方差谨慎解释。所有结论都是条件预测关联，而不是因果效应。

## 12. 运行完整分析

在仓库根目录依次运行：

```bash
uv run python scripts/construct_level_cv.py
uv run python scripts/robustness_measurement.py
uv run python scripts/construct_predictor_profile.py
uv run python scripts/unified_algorithm_benchmark.py
uv run python scripts/ordinal_measurement.py
uv run python scripts/domain_shapley.py
uv run python scripts/cross_model_importance_stability.py
```

初始题项级模型保留为敏感性分析：`uv run python scripts/exploratory_item_level.py`。